# Filter Non-Neonatal ICD Codes for 4 Conditions

This notebook reads the ICD review output and removes **neonatal/newborn** codes.

Input:
- `data/processed/icd_review/icd_codes_for_4_conditions_full.csv`

Output:
- `data/processed/icd_review/icd_codes_for_4_conditions_non_neonatal.csv`

In [1]:
from pathlib import Path
import re

import pandas as pd

BASE_DIR = Path('/home/ubuntu/condition-aware_risk_CDSS')
INPUT_PATH = BASE_DIR / 'data/processed/icd_review/icd_codes_for_4_conditions_full.csv'
OUTPUT_PATH = BASE_DIR / 'data/processed/icd_review/icd_codes_for_4_conditions_non_neonatal.csv'

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Missing input file: {INPUT_PATH}. Run icd_code_validation.ipynb first.')

full_df = pd.read_csv(INPUT_PATH)
full_df['long_title'] = full_df['long_title'].fillna('').astype(str)
full_df['long_title_norm'] = full_df['long_title'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()

print('Loaded:', INPUT_PATH)
print('Rows:', len(full_df))
full_df.head()

Loaded: /home/ubuntu/condition-aware_risk_CDSS/data/processed/icd_review/icd_codes_for_4_conditions_full.csv
Rows: 988


,condition,icd_version,icd_code,long_title,match_source,n_rows,n_subjects,n_hadm,long_title_norm
0,sepsis,9,0380,Streptococcal septicemia,prefix_only,635.0,609.0,635.0,streptococcal septicemia
1,sepsis,9,03810,"Staphylococcal septicemia, unspecified",prefix_only,31.0,31.0,31.0,"staphylococcal septicemia, unspecified"
2,sepsis,9,03811,Methicillin susceptible Staphylococcus aureus ...,prefix_only,490.0,468.0,490.0,methicillin susceptible staphylococcus aureus ...
3,sepsis,9,03812,Methicillin resistant Staphylococcus aureus se...,prefix_only,297.0,262.0,297.0,methicillin resistant staphylococcus aureus se...
4,sepsis,9,03819,Other staphylococcal septicemia,prefix_only,166.0,162.0,166.0,other staphylococcal septicemia


In [2]:
# Neonatal/newborn exclusion terms
neonatal_pattern = re.compile(
    r'newborn|new born|neonat|perinatal|\bpreterm\b|prematur|birth trauma|\binfant\b',
    flags=re.IGNORECASE,
)

full_df['is_neonatal_related'] = full_df['long_title_norm'].apply(lambda x: bool(neonatal_pattern.search(x)))

filtered_df = full_df[~full_df['is_neonatal_related']].copy()

print('Neonatal-related rows removed:', int(full_df['is_neonatal_related'].sum()))
print('Rows remaining:', len(filtered_df))

removed_preview = full_df[full_df['is_neonatal_related']][
    ['condition', 'icd_version', 'icd_code', 'long_title', 'match_source', 'n_rows']
]
print('\nRemoved code preview (up to 30 rows):')
print(removed_preview.head(30).to_string(index=False))

Neonatal-related rows removed: 19
Rows remaining: 969

Removed code preview (up to 30 rows):
condition  icd_version icd_code                                                   long_title match_source  n_rows
   sepsis            9    77181                               Septicemia [sepsis] of newborn   title_only     0.0
   sepsis           10      P36                                  Bacterial sepsis of newborn   title_only     0.0
   sepsis           10     P360              Sepsis of newborn due to streptococcus, group B   title_only     0.0
   sepsis           10     P361  Sepsis of newborn due to other and unspecified streptococci   title_only     0.0
   sepsis           10    P3610            Sepsis of newborn due to unspecified streptococci   title_only     0.0
   sepsis           10    P3619                  Sepsis of newborn due to other streptococci   title_only     0.0
   sepsis           10     P362               Sepsis of newborn due to Staphylococcus aureus   title_only    

In [3]:
# Save filtered codebook and quick summary
save_cols = [
    'condition', 'icd_version', 'icd_code', 'long_title', 'match_source', 'n_rows', 'n_subjects', 'n_hadm'
]
filtered_df[save_cols].to_csv(OUTPUT_PATH, index=False)

summary = (
    filtered_df.groupby(['condition', 'match_source'])['icd_code']
    .count()
    .rename('n_codes')
    .reset_index()
    .sort_values(['condition', 'match_source'])
)

summary_path = OUTPUT_PATH.with_name('icd_codes_for_4_conditions_non_neonatal_summary.csv')
summary.to_csv(summary_path, index=False)

print('Saved filtered file:', OUTPUT_PATH)
print('Saved summary file :', summary_path)
print('\nSummary:')
print(summary.to_string(index=False))

Saved filtered file: /home/ubuntu/condition-aware_risk_CDSS/data/processed/icd_review/icd_codes_for_4_conditions_non_neonatal.csv
Saved summary file : /home/ubuntu/condition-aware_risk_CDSS/data/processed/icd_review/icd_codes_for_4_conditions_non_neonatal_summary.csv

Summary:
    condition match_source  n_codes
          ckd prefix+title       18
          ckd   title_only       50
     diabetes prefix+title      617
     diabetes   title_only      139
heart_failure prefix+title       43
heart_failure  prefix_only        1
heart_failure   title_only       31
       sepsis prefix+title       30
       sepsis  prefix_only       15
       sepsis   title_only       25
